# 🛫 Prévision des Retards Aériens — Notebook MLflow
**Projet PPML- FlyOnTime — 6 plus grands aéroports français**


### Configuration & Imports

In [ ]:
import pandas as pd
import os
import numpy as np
import shap
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import  RandomForestRegressor
from pandas.api.types import is_datetime64_any_dtype
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import mlflow

### Connexion MLflow

In [ ]:
# --- 1. CONFIGURATION SERVEUR ---
MLFLOW_REMOTE_URI = "https://ppml2026-ppml-mlflow.hf.space"
mlflow.set_tracking_uri(MLFLOW_REMOTE_URI)




### Chargement des données



**Datasets :**
- `df_train` → données historiques pour l'entraînement
- `df_predict` → données d'avril 2026 pour la prédiction finale

In [16]:
df_train_final = pd.read_parquet("df_train_final.parquet")

In [17]:
df_train_final.head()

,flight_date,flight_number,airport_origin,airport_destination,departure_delay_min,arrival_delay_min,time_dep,relative_humidity_2m_dep,dew_point_dep,icing_conditions_dep,...,wind_gusts_10m_arr,wind_speed_10m_arr,wind_direction_10m_arr,precipitation_arr,has_precipitation_arr,fog_arr,humidity_arr,temperature_2m_arr,cloud_cover_arr,retard_arrivee
0,2025-09-21,AF 1249,MRS,CDG,9.0,50.0,2025-09-21 07:00:00+00:00,84.0,11.4,Non,...,28.1,14.5,32.0,0.0,Non,Non,85.0,11.6,4.0,1
1,2025-09-21,AF 6004,ORY,MRS,9.0,9.0,2025-09-21 07:00:00+00:00,84.0,11.4,Non,...,22.3,7.6,121.0,0.2,Oui,Non,62.0,22.9,100.0,0
2,2025-09-21,AF 6009,MRS,ORY,109.0,109.0,2025-09-21 14:00:00+00:00,87.0,18.2,Non,...,22.3,7.6,121.0,0.2,Oui,Non,62.0,22.9,100.0,1
3,2025-09-21,AF 6104,ORY,TLS,13.0,0.0,2025-09-21 06:00:00+00:00,84.0,11.4,Non,...,23.0,9.1,321.0,0.0,Non,Non,86.0,16.4,100.0,0
4,2025-09-21,AF 6108,ORY,TLS,11.0,11.0,2025-09-21 10:00:00+00:00,74.0,10.6,Non,...,25.9,10.6,320.0,0.1,Oui,Non,79.0,16.7,100.0,0


In [21]:
df_train_final.columns

Index(['flight_date', 'flight_number', 'airport_origin', 'airport_destination',
       'departure_delay_min', 'arrival_delay_min', 'time_dep',
       'relative_humidity_2m_dep', 'dew_point_dep', 'icing_conditions_dep',
       'rain_dep', 'freezing_rain_dep', 'snow_dep', 'thunderstorms_dep',
       'pressure_msl_dep', 'wind_shear_dep', 'wind_gusts_10m_dep',
       'wind_speed_10m_dep', 'wind_direction_10m_dep', 'precipitation_dep',
       'has_precipitation_dep', 'fog_dep', 'humidity_dep',
       'temperature_2m_dep', 'cloud_cover_dep', 'time_arr',
       'relative_humidity_2m_arr', 'dew_point_arr', 'icing_conditions_arr',
       'rain_arr', 'freezing_rain_arr', 'snow_arr', 'thunderstorms_arr',
       'pressure_msl_arr', 'wind_shear_arr', 'wind_gusts_10m_arr',
       'wind_speed_10m_arr', 'wind_direction_10m_arr', 'precipitation_arr',
       'has_precipitation_arr', 'fog_arr', 'humidity_arr',
       'temperature_2m_arr', 'cloud_cover_arr', 'retard_arrivee'],
      dtype='object')

In [18]:
df_train_final.shape

(92897, 45)

In [20]:
cols_a_virer = ['departure_delay_min','arrival_delay_min','time_dep','time_arr']

### MODELING

In [ ]:
#MANJA
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor
from pandas.api.types import is_datetime64_any_dtype
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================

PRENOM = "Ludo"
TYPE_MODEL = "XGBoost"
EXPERIMENT_NAME = "FlyOnTime_regressor"
RUN_NAME = f"{TYPE_MODEL}_Regressor"
MODEL_NAME = f"{RUN_NAME}_{PRENOM}"
REGISTERED_NAME = f"{RUN_NAME}_registered"
ALIAS_NAME = "challenger"

# =========================================================
# DATA
# =========================================================

# df_train_final est censé être déjà chargé depuis ton parquet
# ex:
# df_train_final = pd.read_parquet("ton_chemin/df_train_final.parquet")

df_train_final_backup = df_train_final.copy()

# garder uniquement les lignes avec target de régression dispo
df_train_final = df_train_final[df_train_final["arrival_delay_min"].notnull()].copy()

# target continue
y = df_train_final["arrival_delay_min"].astype(float)

# features
X = df_train_final.drop(
    columns=cols_a_virer + ["arrival_delay_min", "retard_arrivee"],
    errors="ignore"
).copy()

# =========================================================
# DATETIME CLEAN
# =========================================================

datetime_cols = ["flight_date", "scheduled_departure_dep", "scheduled_arrival_arr"]

def datetime_clean(df, datetime_cols):
    df = df.copy()
    bad_datetime_cols = []

    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

            if not is_datetime64_any_dtype(df[col]):
                bad_datetime_cols.append(col)

    usable_datetime_cols = [
        col for col in datetime_cols
        if col in df.columns and col not in bad_datetime_cols
    ]

    if "flight_date" in usable_datetime_cols:
        df["flight_month"] = df["flight_date"].dt.month
        df["flight_day"] = df["flight_date"].dt.day
        df["flight_dayofweek"] = df["flight_date"].dt.dayofweek

    if "scheduled_departure_dep" in usable_datetime_cols:
        df["sched_dep_hour"] = df["scheduled_departure_dep"].dt.hour
        df["sched_dep_minute"] = df["scheduled_departure_dep"].dt.minute

    if "scheduled_arrival_arr" in usable_datetime_cols:
        df["sched_arr_hour"] = df["scheduled_arrival_arr"].dt.hour
        df["sched_arr_minute"] = df["scheduled_arrival_arr"].dt.minute

    print("Colonnes datetime problématiques droppées :", bad_datetime_cols)

    df = df.drop(columns=datetime_cols, errors="ignore")
    return df

X_processed = datetime_clean(X, datetime_cols)


# =========================================================
# SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
)
# =========================================================
# ENCODAGE APRES SPLIT (important)
# =========================================================

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

if len(cat_cols) > 0:
    X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))
    X_test[cat_cols] = encoder.transform(X_test[cat_cols].astype(str))

# imputation numérique
num_cols_train = X_train.select_dtypes(include=np.number).columns.tolist()

train_medians = X_train[num_cols_train].median()

X_train[num_cols_train] = X_train[num_cols_train].fillna(train_medians)
X_test[num_cols_train] = X_test[num_cols_train].fillna(train_medians)

# sécurité : mêmes colonnes / même ordre
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0

X_test = X_test[X_train.columns]

# =========================================================
# RANDOM SEARCH + CV
# =========================================================

param_dist = {
    "n_estimators": [200, 300, 500, 700],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 1.5, 2, 3]
}

xgb_reg = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# # CV sur y_train binné pour garder un équilibre dans les folds
# y_train_binned = pd.qcut(
#     y_train,
#     q=min(n_bins, y_train.nunique()),
#     duplicates="drop"
# )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_reg,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_mean_absolute_error",
    cv=cv.split(X_train, y_train),  # utiliser y_train_binned si stratified
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

# =========================================================
# MLFLOW + TRAIN
# =========================================================

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tag("developer", PRENOM)
    mlflow.set_tag("model_full_name", MODEL_NAME)
    mlflow.set_tag("target", "arrival_delay_min")
    mlflow.set_tag("model_type", "regression")

    print(f"Entraînement {TYPE_MODEL} Regressor avec CV + Random Search...")
    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_

    # =====================================================
    # PREDICTIONS
    # =====================================================

    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)

    # =====================================================
    # METRICS
    # =====================================================

    train_metrics = {
        "train_mae": mean_absolute_error(y_train, y_pred_train),
        "train_rmse": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "train_r2": r2_score(y_train, y_pred_train)
    }

    test_metrics = {
        "test_mae": mean_absolute_error(y_test, y_pred_test),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "test_r2": r2_score(y_test, y_pred_test)
    }

    all_metrics = {**train_metrics, **test_metrics}

    # log params
    mlflow.log_params(random_search.best_params_)
    mlflow.log_metric("best_cv_score_neg_mae", random_search.best_score_)

    # log metrics
    mlflow.log_metrics(all_metrics)

    # log model
    model_info = mlflow.xgboost.log_model(
        xgb_model=best_model,
        name=MODEL_NAME,
        registered_model_name=REGISTERED_NAME,
        input_example=X_test.head(5),
        model_format="json"
    )

    # alias model registry
    client = mlflow.tracking.MlflowClient()
    client.set_registered_model_alias(
        REGISTERED_NAME,
        ALIAS_NAME,
        str(model_info.registered_model_version)
    )

    # =====================================================
    # AFFICHAGE FINAL
    # =====================================================

    print("\n===== BEST PARAMS =====")
    print(random_search.best_params_)

    print("\n===== METRICS TRAIN =====")
    for name, value in train_metrics.items():
        print(f"{name}: {value:.4f}")

    print("\n===== METRICS TEST =====")
    for name, value in test_metrics.items():
        print(f"{name}: {value:.4f}")

    print(f"\nModèle enregistré : {REGISTERED_NAME} @{ALIAS_NAME}")

Colonnes datetime problématiques droppées : []
Entraînement XGBoost Regressor avec CV + Random Search...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
